In [79]:
from dataclasses import dataclass
from datetime import date
import os
from pathlib import Path
import tomllib
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine
from nvi_etl.geo_reference import (
    pull_city_boundary, 
    pull_council_districts, 
    pull_zones, 
    pull_cdo_boundaries,
)
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows


WORKING_DIR = Path(os.getcwd()).parent.parent


with open(WORKING_DIR / "config.toml", "rb") as f:
    config = tomllib.load(f)


def get_engine(db_name=None):
    user = config["db"]["user"]
    password=config["db"]["password"]
    database=db_name or config["db"]["database"]
    host=config["db"]["host"]

    return create_engine(f"postgresql+psycopg://{user}:{password}@{host}:5432/{database}")

In [44]:
districts = pull_council_districts(2026)
zones = pull_zones(2026)

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [ ]:
def combine_survey_and_geocoded(frame, geoframe):
    return gpd.GeoDataFrame(
        frame.rename(
            columns={
                "Response ID": "response_id",
            }
        ).merge(
            geoframe.rename(
                columns={
                    "Response_I": "response_id",
                }
            )[["response_id", "geometry"]], 
            on="response_id"
        )
    )

In [81]:
frame = pd.read_csv(
    "Q:\\3_Projects\\NVI\\2025\\nvi_survey_data_2025_20260226.csv", 
    low_memory=False
).rename(columns={"Response ID": "response_id"})

geoframe = gpd.read_file(
    "Q:\\3_Projects\\NVI\\2025\\Final Shapefiles\\Final2025NVIDataset_cleaned_20260304.shp"
).rename(columns={"Response_I": "response_id"})

datadictionary = pd.read_excel(WORKING_DIR / "primary_survey" / "v2025" / "conf" / "nvi_answer_key_20260316.xlsx")

location_dictionary = pd.read_excel(
    WORKING_DIR / "primary_survey" / "v2025" / "conf" / "locations_20260312.xlsx", dtype={"location": str}
)

geocoded = combine_survey_and_geocoded(frame, geoframe)

# # NOTE SC: Commented out joins for zones and districts for now 
# # NOTE MV: I've uncommented, because we can allow this to flow into the full response, 
# # and then filter later. We have plans to include the CDOs in the database.
geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(districts[["geometry", "district_number"]], how="left", predicate="within")
    .drop(columns="index_right")
)

geocoded = (
    geocoded
    .to_crs(2898)
    .sjoin(zones[["geometry", "zone_id"]], how="left", predicate="within")
)

In [ ]:
# Let's make sense sure we didn't lose any rows
assert len(frame) == len(geocoded)

In [ ]:
@dataclass
class Table:
    topic: str | None
    question: str 
    table: pd.DataFrame | None = None
    summary_level: str | None = None


def make_table(column, table=None, summary_level=None):
    return Table(
        column["topic_text"] or None,
        column["question_text"],
        table,
        summary_level=summary_level,
    )

In [ ]:
result = {}
serious = []
errors = []
# Loop through each question
for i, column in datadictionary.drop_duplicates(subset="full_column").iterrows():
    if (
        (column["start_date"] > pd.Timestamp(date(year=2026, month=1, day=1)))
        or (column["end_date"] < pd.Timestamp(date(year=2026, month=1, day=1)))
    ):
        continue

    tables = []

    topic_text = column["topic_text"]
    column_name = column["full_column"].replace("'", "’")

    if column_name not in frame:
        print(column["indicator_db_id"])
        errors.append(column_name)
        continue

    labels = datadictionary[
        datadictionary["full_column"] == column_name
    ][["survey_code", "answer"]].set_index("survey_code")

    # First aggregate city-wide
    out = (
        geocoded[column_name]
        .value_counts(dropna=False)
        .to_frame()
    )

    try:
        out.index = out.index.astype(pd.Int64Dtype())
    except ValueError:
        errors.append("citywide")
        continue

    out = (
        out
        .join(labels)
        .sort_values(column_name)
        .reset_index(drop=True)
    )[["answer", "count"]]

    tables.append(make_table(column, out, "citywide"))


    # Then aggregate districts, zones

    for summary in ("district_number", "zone_id"):
        out = (
            geocoded[[column_name, summary]]
            .value_counts(dropna=False)
            .reset_index(summary)
        )

        out.index = out.index.astype(pd.Int64Dtype())

        out = (
            out
            .join(labels)
            .sort_values([summary, column_name])
            .reset_index(drop=True)
        )[[summary, "answer", "count"]]

        tables.append(make_table(column, out, summary))

    result[column["question_text"]] = tables

In [ ]:
wb = Workbook()
toc = wb.active
toc.title = "Table of Contents"
toc.append(["Sheet #", "Question"])

for sheet_num, (question, tables) in enumerate(result.items(), 1):
    ws = wb.create_sheet(title=f"Q{sheet_num}")
    toc.append([sheet_num, question])
    first = tables[0]
    if first.topic:
        ws.append([first.topic])
    ws.append([first.question])
    for table in tables:
        if table.summary_level:
            ws.append([table.summary_level])
        for r in dataframe_to_rows(table.table, index=False, header=True):
            ws.append(r)
        ws.append([])

wb.save("survey_output_20260312.xlsx")

In [ ]:
def append_universe_and_percentages(table):
    included = table["universe_include"]

    # Calculate a percentage column
    total = (
        table[included]
        .groupby("location")["count"]
        .transform("sum")
    )

    table["universe"] = total
    table.loc[included, "percentage"] = table.loc[included, "count"] / total

    return table


def aggregate_group():
    pass


def aggregate_question(frame: pd.DataFrame, column_name: str, 
    summary: str, labels: pd.DataFrame) -> pd.DataFrame:


    aggregation_type = labels.iloc[0]["response_type"]
    """
    This assumes question is answered with a selection from a likert item


    in:
        frame: our current dataframe
        column_name: the name of the frame we're aggregating
        summary: the summary level we're working with -- another column
        labels: rows of the datadictionary associated with the column name

    out:
        The rows aggregated
    """
    table = (
        frame[[column_name, summary]]
        .value_counts(dropna=False)
        .reset_index(summary)
    )

    # Reference for filling in join misses
    reference = labels.iloc[0]

    table.index = table.index.astype(pd.Int64Dtype())

    table = (
        table
        .join(labels, how="outer")
        .sort_values([summary, column_name])
        .reset_index(drop=True)
        .rename(columns={summary: "location"})
        .fillna({
            "topic_text": reference["topic_text"],
            "question_text": reference["question_text"],
            "full_column": reference["full_column"],
            "answer": "[SKIPPED]",
            "db_question_code": reference["db_question_code"],
            "indicator_include": False,
            "universe_include": False,
            "indicator_db_id": reference["indicator_db_id"],
            "location": "[NO ADDRESS PROVIDED]",
        })
        .astype({
            "db_question_code": pd.Int64Dtype(), 
            "db_answer_code": pd.Int64Dtype(),
            "indicator_db_id": pd.Int64Dtype(),
            "universe_include": bool,
            "indicator_include": bool,
        })
        .assign(summary_level=summary)
    )

    return append_universe_and_percentages(table)

In [ ]:
# Try likert -- checks out

column_name = "Stay_12_Months:Agreement_Statements_Access_and_Support"
labels = datadictionary[datadictionary["full_column"] == column_name]
summary = "district_number"

aggregate_question(geocoded, column_name, summary, labels).columns

C:\Users\mike\AppData\Local\Temp\ipykernel_34028\3438340802.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna({


Index(['location', 'count', 'indicator_db_id', 'indicator_include',
       'universe_include', 'group', 'survey_question_topic_id', 'topic_text',
       'question_text', 'answer', 'survey_code', 'db_question_code',
       'db_answer_code', 'tabulate', 'suppress_value', 'full_column',
       'response_type', 'universe_query', 'site_category', 'and_or',
       'start_date', 'end_date', 'summary_level', 'universe', 'percentage'],
      dtype='object')

In [ ]:
column_name = "Stay_12_Months:Agreement_Statements_Access_and_Support"
labels = datadictionary[datadictionary["full_column"] == column_name]
summary = "district_number"

In [ ]:
out_columns = [
    'location', 'count', 'indicator_db_id', 'indicator_include',
    'universe_include', 'group', 'survey_question_topic_id', 'topic_text',
    'question_text', 'answer', 'survey_code', 'db_question_code',
    'db_answer_code', 'tabulate', 'suppress_value', 'full_column',
    'response_type', 'universe_query', 'site_category', 'and_or',
    'start_date', 'end_date', 'summary_level', 'universe', 'percentage'
]

In [ ]:
group_name = "Programs_Affordable_Internet"
labels = datadictionary[datadictionary["group"] == group_name]
summary = "district_number"

result = []
for summary in ("district_number", "zone_id"):

    # Function aggregate_multiselect will start here
    groups = geocoded.groupby(summary)
    universe = groups.size().rename("universe").reset_index() # all rows with value regardless of value

    result.append(
        groups[labels["full_column"]]
        .count()
        .melt(ignore_index=False, var_name="full_column")
        .reset_index()
        .merge(
            labels[[
                "full_column", 
                "db_question_code", 
                "db_answer_code",
                'and_or',
                'answer',
                'end_date',
                'group',
                'indicator_db_id',
                'indicator_include',
                'question_text',
                'response_type',
                'site_category',
                'start_date',
                'suppress_value',
                'survey_code',
                'survey_question_topic_id',
                'tabulate',
                'topic_text',
                'universe_include',
                'universe_query'
            ]], 
            on="full_column"
        )
        .merge(
            universe,
            on=summary
        )
        .rename(columns={
            "value": "count",
            summary: "location",
        })
        .assign(
            percentage=lambda frame: (100 * frame["count"] / frame["universe"]).round(2),
            summary_level=summary,
        )
        .astype({
            "db_question_code": pd.Int64Dtype(),
            "db_answer_code": pd.Int64Dtype(),
        })
    )

table = pd.concat(result)

pd.Index(out_columns).difference(table.columns)

Index(['percentage', 'summary_level'], dtype='object')

In [ ]:
new_frame = pd.read_csv("test_output_20260317.csv")
org_frame = pd.read_csv("test_output_20260317_orig.csv")

In [ ]:
assert len(new_frame) == len(org_frame)

sort_columns = ["location_id", "topic_text", "indicator_db_id", "db_question_code", "response_type", "db_answer_code"]


new_frame = new_frame[
    (new_frame.location != "[NO ADDRESS PROVIDED]")
    & (new_frame.answer != "[SKIPPED]")
] 
org_frame = org_frame[
    (org_frame.location != "[NO ADDRESS PROVIDED]")
    & (org_frame.answer != "[SKIPPED]")
] 


incorrect = (
    new_frame.sort_values(sort_columns).reset_index(drop=True) 
    != org_frame.sort_values(sort_columns).reset_index(drop=True)
)

In [ ]:
len(incorrect)

16717

In [ ]:
new_frame.reset_index()[incorrect.any(axis=1)]

,index,location,location_id,summary_level,full_column,response_type,db_question_code,indicator_db_id,topic_text,question_text,indicator_include,universe_include,db_answer_code,answer,count,universe,percentage
35,49,2,3.0,district_number,Computer:Access_Device,GROUPED-SINGLE,118,90.0,Do you have access to one or more of these dev...,Desktop or laptop computer,False,True,90.0,I don't have access,193.0,968.0,0.199380
36,51,3,4.0,district_number,Computer:Access_Device,GROUPED-SINGLE,118,90.0,Do you have access to one or more of these dev...,Desktop or laptop computer,True,True,88.0,My own,409.0,644.0,0.635093
37,52,3,4.0,district_number,Computer:Access_Device,GROUPED-SINGLE,118,90.0,Do you have access to one or more of these dev...,Desktop or laptop computer,True,True,89.0,Share it,60.0,644.0,0.093168
38,53,3,4.0,district_number,Computer:Access_Device,GROUPED-SINGLE,118,90.0,Do you have access to one or more of these dev...,Desktop or laptop computer,False,True,90.0,I don't have access,175.0,644.0,0.271739
39,55,4,5.0,district_number,Computer:Access_Device,GROUPED-SINGLE,118,90.0,Do you have access to one or more of these dev...,Desktop or laptop computer,True,True,88.0,My own,708.0,1045.0,0.677512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16712,20165,7a,29.0,zone_id,Youth_in_Household_YesNo,SINGLE,167,NaN,NaN,Do you have young people under the age of 18 i...,False,True,37.0,No,187.0,306.0,0.611111
16713,20167,7b,30.0,zone_id,Youth_in_Household_YesNo,SINGLE,167,NaN,NaN,Do you have young people under the age of 18 i...,False,True,36.0,Yes,60.0,177.0,0.338983
16714,20168,7b,30.0,zone_id,Youth_in_Household_YesNo,SINGLE,167,NaN,NaN,Do you have young people under the age of 18 i...,False,True,37.0,No,117.0,177.0,0.661017
16715,20169,7c,31.0,zone_id,Youth_in_Household_YesNo,SINGLE,167,NaN,NaN,Do you have young people under the age of 18 i...,False,True,36.0,Yes,116.0,338.0,0.343195


In [ ]:

# pd.read_csv()
os.listdir()

['1_projects',
 '2_responsibilities',
 '3_worker_ownership',
 '4_library',
 '5_archive',
 '6_lab',
 'desktop.ini',
 'survey_question.csv',
 'survey_question_topic.csv',
 '~$digital_equity_questions_20250512.xlsx']

In [ ]:
frame = pd.read_csv(
    Path.home() / "Desktop" / "survey_question.csv", encoding="latin-1"
)

In [ ]:
frame

,id,question_text,question_topic_id,question_type_id
0,1,I currently participate in a block/ neighborho...,1.0,4
1,2,I currently participate in a community group o...,1.0,4
2,4,"In the last 12 months, I have participated in ...",NaN,3
3,5,"In the last 12 months, I haveâ¦ (Select all t...",NaN,3
4,6,Housing resources and supports (such as evicti...,2.0,5
...,...,...,...,...
109,111,To what extent do you agree that opportunities...,NaN,4
110,112,"Overall, how satisfied are you with opportunit...",NaN,4
111,113,How satisfied are you that the opportunities f...,NaN,4
112,114,"Within the past 12 months at work, do you feel...",NaN,4


In [ ]:
(
    new_frame[["db_question_code", "question_text"]]
    .drop_duplicates()
    .sort_values("db_question_code")
    .to_csv(Path.home() / "Desktop" / "new_question.csv")
)

In [ ]:
new_frame.db_question_code.drop_duplicates()

0        117
4        118
8        119
380        1
385        2
        ... 
19330    116
19517     53
19523     54
19882    168
20092    167
Name: db_question_code, Length: 107, dtype: int64

In [57]:
from sqlalchemy.dialects.postgresql import insert

def upsert(table, conn, keys, data_iter):
    data = [dict(zip(keys, row)) for row in data_iter]
    stmt = insert(table.table).values(data)
    stmt = stmt.on_conflict_do_update(
        index_elements=["id"],  # your unique/pk column(s)
        set_={k: stmt.excluded[k] for k in keys if k != "id"},
    )
    conn.execute(stmt) 

topics = (
    datadictionary[["survey_question_topic_id", "topic_text"]]
    .drop_duplicates()
    .dropna()
    .astype({"survey_question_topic_id": pd.Int64Dtype()})
    .sort_values("survey_question_topic_id")
    .rename(
        columns={
            "survey_question_topic_id": "id",
            "topic_text": "name",
        }
    )
)

topics.to_sql(
    "survey_question_topic", get_engine("nvi_test"), 
    if_exists="append", method=upsert, index=False
)

In [82]:
questions = (
    datadictionary[[
        "db_question_code", 
        "question_text", 
        "survey_question_topic_id", 
        "survey_question_type"
    ]]
    .drop_duplicates()
    .dropna(subset="db_question_code")
    .astype({
        "survey_question_topic_id": pd.Int64Dtype(),
        "db_question_code": pd.Int64Dtype(),
    })
    .sort_values(["db_question_code", "survey_question_topic_id"])
    .rename(
        columns={
            "db_question_code": "id",
            "survey_question_topic_id": "question_topic_id",
            "survey_question_type": "question_type_id",
        }
    )
    .drop_duplicates(subset="id", keep="last")
    .dropna(subset="question_text")
)

questions.to_sql(
    "survey_question", get_engine("nvi_test"), 
    if_exists="append", method=upsert, index=False
)

In [90]:
(
    datadictionary[["db_answer_code", "answer"]]
    .drop_duplicates()
    .astype({"db_answer_code": pd.Int64Dtype()})
    .rename(columns={"db_answer_code": "id", "answer": "name"})
    .to_clipboard(index=False)
)

In [ ]:
answers = pd.read_excel(
    "P:\\2025_Projects\\NVI25\\Development\\Workspace\\nvi_answer_ids_new_20260318.xlsx")

answers[["id", "name", "set_id", "sort_order"]].to_sql(
    "survey_question_option", get_engine("nvi_test"),
    if_exists="append", method=upsert, index=False,
)